1. Add a third candidate to section 8's comparison (e.g. `Ridge` or `GradientBoostingRegressor`) and rank all three by CV score.

In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

california = fetch_california_housing(as_frame=True)
housing = california.frame

rng = np.random.default_rng(42)
messy = housing.copy()
missing_idx = rng.choice(messy.index, size=200, replace=False)
messy.loc[missing_idx, "AveBedrms"] = np.nan
missing_idx_2 = rng.choice(messy.index, size=120, replace=False)
messy.loc[missing_idx_2, "AveRooms"] = np.nan

messy["coast_region"] = pd.cut(
    messy["Longitude"],
    bins=[-125, -121.22, -118.20, -114],
    labels=["Coastal", "Central", "Inland"],
)
messy["income_cat"] = pd.cut(
    messy["MedInc"],
    bins=[0, 2.5, 4.5, 7, np.inf],
    labels=["Low", "Medium", "High", "Very High"],
)
messy["bedrooms_ratio"] = messy["AveBedrms"] / messy["AveRooms"]
messy["rooms_per_person"] = messy["AveRooms"] / messy["AveOccup"]

x = messy.drop(columns="MedHouseVal")
y = messy["MedHouseVal"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

numeric_features = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude", "Longitude"]
categorical_features = ["coast_region"]

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression

cat_pipe=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoded",OneHotEncoder())
])
numeric_pipe=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])
preprocess=ColumnTransformer([
    ("cat",cat_pipe,categorical_features),
    ("numeric",numeric_pipe,numeric_features)
])

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

models={
    "linier":LinearRegression(),
    "forest":RandomForestRegressor(n_estimators=100,n_jobs=-1,random_state=42),
    "gradian":GradientBoostingRegressor(n_estimators=100,random_state=42)
}

cv_scores=[]

for name,model in models.items():
    pipe=Pipeline([("pre",preprocess),("model",model)])
    pipe.fit(x_train,y_train)
    cv=cross_val_score(pipe,x,y,cv=5,scoring="neg_root_mean_squared_error")
    cv_scores.append(cv.mean())

In [15]:
cv_scores

[np.float64(-0.7488198742283558),
 np.float64(-0.6511434358887023),
 np.float64(-0.6466604083452949)]

2. Widen section 10's `param_grid` with one more value per hyperparameter; does `best_params_` change?

In [16]:
from sklearn.model_selection import GridSearchCV

forest_pipeline = Pipeline([
    ("preprocessor", preprocess),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
])

param_grid = {
    "model__n_estimators": [50,100, 200, 300],
    "model__max_depth": [None,5, 10, 20],
}

grid_search = GridSearchCV(
    forest_pipeline, param_grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
)
grid_search.fit(x_train, y_train)
grid_search.best_params_

{'model__max_depth': None, 'model__n_estimators': 300}

3. Re-run section 11 with `n_iter=20` instead of 10; how close does `best_score_` get to section 10's grid result?

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "model__n_estimators": [50, 100, 150, 200, 250, 300, 400],
    "model__max_depth": [None, 5, 10, 15, 20, 25],
}

random_search = RandomizedSearchCV(
    forest_pipeline, param_distributions, n_iter=20, cv=5,
    scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1,
)
random_search.fit(x_train, y_train)
-random_search.best_score_